### Завдання 1. 
Завантажити та відкрити (вручну або через запропонований скрипт на сайті) наступний датасет: Individual Household Electric Power Consumption Dataset. Датасет знаходиться у файлі household_power_consumption.txt

In [1]:
import pandas as pd
import numpy as np
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from IPython.display import display

print('Завантажуємо датасет та перетворюємо символи "?" на порожні значення та видаляємо їх.')
HPS = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?', low_memory=False)
HPS.dropna(inplace=True)

print(f"Дані очищено. Маємо таку кількість записів: {len(HPS)}")

Завантажуємо датасет та перетворюємо символи "?" на порожні значення та видаляємо їх.
Дані очищено. Маємо таку кількість записів: 2049280


### Завдання 2. Реалізувати процедури та здійснити профілювання часу виконання за допомогою модуля timeit.
##### 2.1. Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.

In [2]:
def get_high_power_records(data):
    return data[data['Global_active_power'] > 5]

display(get_high_power_records(HPS).head(3))

print('\n Час виконання функції (10 повторень, в секундах)')
execution_time = timeit.timeit(lambda: get_high_power_records(HPS), number=10)
print(execution_time)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0



 Час виконання функції (10 повторень, в секундах)
0.05190720000246074


##### 2.2. Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).

In [3]:
def filter_by_appliances_and_current(dataframe):
    condition_min = dataframe['Global_intensity'] >= 19.0
    condition_max = dataframe['Global_intensity'] <= 20.0
    condition_appliances = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    return dataframe[condition_min & condition_max & condition_appliances]

print("Результат вибірки:")
result_appliances = filter_by_appliances_and_current(HPS)
display(result_appliances.head(3))

print('\n Час виконання функції (10 повторень, в секундах)')
execution_time = timeit.timeit(lambda: filter_by_appliances_and_current(HPS), number=10)
print(execution_time)

Результат вибірки:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0



 Час виконання функції (10 повторень, в секундах)
0.1000686999905156


##### 2.3. Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [4]:
def get_random_sample_and_means(dataframe, sample_size=500000):

    random_sample = dataframe.sample(n=sample_size, replace=False)
    
    means = random_sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    
    return random_sample, means

print("Випадкова вибірка 500 000 записів та обчислення середніх величин")

sample_df, average_consumption = get_random_sample_and_means(HPS)

print("Середні величини споживання для вибірки:")
print(average_consumption)

print("\n Час виконання (секунди, 10 повторень):")
execution_time = timeit.timeit(lambda: get_random_sample_and_means(HPS), number=10)
print(execution_time)

Випадкова вибірка 500 000 записів та обчислення середніх величин
Середні величини споживання для вибірки:
Sub_metering_1    1.128336
Sub_metering_2    1.292608
Sub_metering_3    6.463982
dtype: float64

 Час виконання (секунди, 10 повторень):
2.1092675999971107


##### 2.4. Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [5]:
def filter_smart_sampling(dataframe):
    condition = (dataframe['Time'] >= '18:00:00') & \
                (dataframe['Global_active_power'] > 6.0) & \
                (dataframe['Sub_metering_2'] > dataframe['Sub_metering_1']) & \
                (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])
    
    res = dataframe[condition]
    
    n = len(res)
    if n < 2: 
        return res
    
    mid = n // 2
    
    # [start : stop : step]
    first_half = res.iloc[:mid:3]   
    second_half = res.iloc[mid::4]  
    
    return pd.concat([first_half, second_half])

print("Результат вибірки (через рядкове порівняння):")
result = filter_smart_sampling(HPS)
display(result.head())

print("\n Час виконання (секунди, 10 повторень):")
execution_time = timeit.timeit(lambda: filter_smart_sampling(HPS), number=10)
print(execution_time)

Результат вибірки (через рядкове порівняння):


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0



 Час виконання (секунди, 10 повторень):
1.3401729999895906


### Завдання 3.
Пронормувати та стандартизувати вибраний датасет

In [6]:
print("Візьмемо копію нашого датафрейму")
numeric_col = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                   'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

hps_copy = HPS[numeric_col].copy()

minmax = MinMaxScaler()
normalized = pd.DataFrame(
    minmax.fit_transform(hps_copy), 
    columns=numeric_col
)

standard = StandardScaler()
standardized = pd.DataFrame(
    standard.fit_transform(hps_copy), 
    columns=numeric_col
)

print("Пронормовані дані (0 до 1):")
display(normalized.head(5))

print("Стандартизовані (середнє = 0, дисперсія = 1):")
display(standardized.head(5))

Візьмемо копію нашого датафрейму
Пронормовані дані (0 до 1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


Стандартизовані (середнє = 0, дисперсія = 1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955077,2.610721,-1.851816,3.098789,-0.182337,-0.051274,1.249421
1,4.037085,2.770406,-2.225274,4.133800,-0.182337,-0.051274,1.130897
2,4.050326,3.320432,-2.330213,4.133800,-0.182337,0.120487,1.249421
3,4.063567,3.355917,-2.191324,4.133800,-0.182337,-0.051274,1.249421
4,2.434881,3.586573,-1.592556,2.513782,-0.182337,-0.051274,1.249421


### Завдання 4.
Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів. Я обрав силу струму (Global_intensity) та активну потужність (Global_active_power), адже вони фізично пропорційні за формулою у фізиці P=U*I, через це й коефіцієнт буде дорівнювати майже 1.

In [7]:
col1 = 'Global_intensity'
col2 = 'Global_active_power'

pearson_val = HPS[col1].corr(HPS[col2], method='pearson')
spearman_val = HPS[col1].corr(HPS[col2], method='spearman')

print(f"Аналіз залежності між {col1} та {col2}:")
print(f"Коефіцієнт Пірсона: {pearson_val:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_val:.4f}")

Аналіз залежності між Global_intensity та Global_active_power:
Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


### Завдання 5.
Провести One Hot Encoding категоріального атрибута. Створюю категоріальний атрибут "Time_Category", після чого створюю стовпці кожної пори дня.

In [8]:
def encode_time_of_day_slim(dataframe):
    hours = pd.to_datetime(dataframe['Time'], format='%H:%M:%S').dt.hour
    
    bins = [0, 6, 12, 18, 24]
    labels = ['Night', 'Morning', 'Afternoon', 'Evening']
    
    category_series = pd.cut(hours, bins=bins, labels=labels, include_lowest=True)
    
    encoded = pd.get_dummies(category_series, prefix='Category')
    
    result = pd.concat([dataframe['Time'], category_series.rename('Time_Category'), encoded], axis=1)
    
    return result

encoded_result = encode_time_of_day_slim(HPS)
display(encoded_result.head(100))

,Time,Time_Category,Category_Night,Category_Morning,Category_Afternoon,Category_Evening
0,17:24:00,Afternoon,False,False,True,False
1,17:25:00,Afternoon,False,False,True,False
2,17:26:00,Afternoon,False,False,True,False
3,17:27:00,Afternoon,False,False,True,False
4,17:28:00,Afternoon,False,False,True,False
...,...,...,...,...,...,...
95,18:59:00,Afternoon,False,False,True,False
96,19:00:00,Evening,False,False,False,True
97,19:01:00,Evening,False,False,False,True
98,19:02:00,Evening,False,False,False,True
